# BT4012 Fraud Analytics Kaggle Competition

**Student:** e1120977

## Aim

The task is to predict the probability that a Bitcoin transaction is illicit. The main challenge is that the training file contains many unknown labels and the test transactions belong to later time steps. Because the Kaggle metric is ROC-AUC, I focus on producing a useful ranking of transactions rather than choosing a single classification threshold.

My approach is fairly direct: start with the supplied features, add a small number of graph features from the edge list, use a chronological validation split, and compare logistic regression against a few LightGBM models.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

SEED = 42
DATA_DIR = Path("data")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
edges = pd.read_csv(DATA_DIR / "txs_edgelist.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:", train.shape)
print("test:", test.shape)
print("edges:", edges.shape)
print("sample submission:", sample.shape)


## 1. Quick data check

The training file has both labelled and unknown rows. I only use rows with a known `label` for supervised learning. The unknown rows are not treated as class 0 because that would add a large amount of noisy negative data.

I also check the time range because the competition is a future-time prediction problem.

In [ ]:
known = train[train["label"].notna()].copy()
print("Known labels:", len(known))
print("Unknown labels:", train["label"].isna().sum())
print("Label counts:")
print(known["label"].value_counts().sort_index())
print("\nTrain time steps:", train["time_step"].min(), "to", train["time_step"].max())
print("Test time steps:", test["time_step"].min(), "to", test["time_step"].max())

assert set(sample.columns) == {"index", "target"}
assert len(sample) == len(test)


## 2. Graph feature engineering

The edge list gives directed flows between transaction IDs. Rather than build a full graph model, I use a few simple node-level summaries. These are easy to interpret and add little computational overhead:

- number of incoming edges
- number of outgoing edges
- unique incoming neighbours
- unique outgoing neighbours
- total degree
- out-degree minus in-degree
- in-degree divided by `(out-degree + 1)`

The features use the supplied graph only; no labels are used in constructing them. Missing degree counts mean that a transaction does not appear in that direction, so I fill those values with zero.

In [ ]:
in_degree = edges["txId2"].value_counts().rename("in_degree")
out_degree = edges["txId1"].value_counts().rename("out_degree")
unique_in = edges.groupby("txId2")["txId1"].nunique().rename("unique_in_neighbors")
unique_out = edges.groupby("txId1")["txId2"].nunique().rename("unique_out_neighbors")

def add_graph_features(df):
    out = df.copy()
    for s in [in_degree, out_degree, unique_in, unique_out]:
        out = out.merge(s, left_on="txId", right_index=True, how="left")
    gcols = ["in_degree", "out_degree", "unique_in_neighbors", "unique_out_neighbors"]
    out[gcols] = out[gcols].fillna(0)
    out["total_degree"] = out["in_degree"] + out["out_degree"]
    out["degree_gap"] = out["out_degree"] - out["in_degree"]
    out["in_out_ratio"] = out["in_degree"] / (out["out_degree"] + 1.0)
    return out

known = add_graph_features(known)
test_model = add_graph_features(test)

feature_cols = [c for c in known.columns if c.startswith("feat_")]
graph_cols = [
    "in_degree", "out_degree", "unique_in_neighbors",
    "unique_out_neighbors", "total_degree", "degree_gap", "in_out_ratio"
]
feature_cols += graph_cols

known[feature_cols] = known[feature_cols].replace([np.inf, -np.inf], np.nan)
test_model[feature_cols] = test_model[feature_cols].replace([np.inf, -np.inf], np.nan)

print("Number of model features:", len(feature_cols))
print("Graph features:", graph_cols)


## 3. Validation setup

I use time steps 1-30 for fitting and 31-35 for validation. This is deliberately not a random split. A random split would mix earlier and later transactions and make the validation setting less similar to the competition test set, which starts at time step 36.

In [ ]:
fit_df = known[known["time_step"] <= 30].copy()
val_df = known[known["time_step"] >= 31].copy()

X_fit = fit_df[feature_cols]
y_fit = fit_df["label"].astype(int)
X_val = val_df[feature_cols]
y_val = val_df["label"].astype(int)

print("Fit rows:", len(fit_df), "positive rate:", round(y_fit.mean(), 4))
print("Validation rows:", len(val_df), "positive rate:", round(y_val.mean(), 4))


## 4. Baseline: logistic regression

I use median imputation because the model cannot take missing values directly, together with class balancing. Logistic regression is useful here as a simple linear reference point; I do not expect it to capture the non-linear interactions in the anonymised transaction features.

In [ ]:
logit = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(
        max_iter=300, class_weight="balanced", C=0.5, solver="liblinear", random_state=SEED
    )
)
logit.fit(X_fit, y_fit)
logit_pred = logit.predict_proba(X_val)[:, 1]
logit_auc = roc_auc_score(y_val, logit_pred)
print(f"Logistic regression ROC-AUC: {logit_auc:.6f}")


## 5. LightGBM experiments

I compared a few small LightGBM variations rather than doing a very large hyperparameter search. The main changes are tree size, learning rate, minimum child size and regularisation. I also keep subsampling and feature subsampling below 1 to reduce the chance of fitting noise.

In [ ]:
lgbm_configs = [
    {
        "n_estimators": 500, "learning_rate": 0.04, "num_leaves": 31,
        "min_child_samples": 40, "subsample": 0.90, "colsample_bytree": 0.80,
        "reg_lambda": 1.0
    },
    {
        "n_estimators": 650, "learning_rate": 0.03, "num_leaves": 24,
        "min_child_samples": 60, "subsample": 0.90, "colsample_bytree": 0.80,
        "reg_lambda": 2.0
    },
    {
        "n_estimators": 650, "learning_rate": 0.03, "num_leaves": 16,
        "min_child_samples": 80, "subsample": 0.90, "colsample_bytree": 0.80,
        "reg_lambda": 3.0
    },
]

results = []
models = []
for i, cfg in enumerate(lgbm_configs, start=1):
    model = LGBMClassifier(
        objective="binary", random_state=SEED, n_jobs=-1, verbosity=-1, **cfg
    )
    model.fit(X_fit, y_fit)
    pred = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, pred)
    results.append({"model": f"LightGBM {i}", "roc_auc": auc})
    models.append(model)
    print(f"LightGBM {i}: {auc:.6f}")

results_df = pd.DataFrame(results)
results_df = pd.concat(
    [pd.DataFrame([{"model": "Logistic regression", "roc_auc": logit_auc}]), results_df],
    ignore_index=True
)
results_df.sort_values("roc_auc", ascending=False)


In [ ]:
best_cfg = lgbm_configs[0]
best_model = models[0]

feature_importance = (
    pd.Series(best_model.feature_importances_, index=feature_cols)
    .sort_values(ascending=False)
    .head(15)
)
print(feature_importance)

plt.figure(figsize=(7, 5))
feature_importance.sort_values().plot(kind="barh")
plt.title("Top 15 LightGBM feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 6. Final model and submission

The first LightGBM configuration had the best validation ROC-AUC (0.998351 in my run). I refit that configuration on all known labels, including the later labelled rows, because the final test period comes after the training period.

I average predictions from two random seeds. This is a small ensemble rather than a large model search and mainly reduces sensitivity to the particular LightGBM seed.

The output is kept as probabilities because ROC-AUC evaluates the ranking of the predictions.

In [ ]:
X_all = known[feature_cols]
y_all = known["label"].astype(int)
X_test = test_model[feature_cols]

preds = []
for seed in [42, 2026]:
    final_model = LGBMClassifier(
        objective="binary", random_state=seed, n_jobs=-1, verbosity=-1, **best_cfg
    )
    final_model.fit(X_all, y_all)
    preds.append(final_model.predict_proba(X_test)[:, 1])

test_pred = np.mean(preds, axis=0)
submission = pd.DataFrame({"index": test["index"], "target": test_pred})

assert list(submission.columns) == ["index", "target"]
assert len(submission) == len(test)
assert np.isfinite(submission["target"]).all()
assert ((submission["target"] >= 0) & (submission["target"] <= 1)).all()

submission.to_csv("submission.csv", index=False)
print(submission.head())
print("\nSaved submission.csv with", len(submission), "rows")
print("Prediction range:", float(test_pred.min()), "to", float(test_pred.max()))


## 7. Final checks

The submission file should have exactly the same row count and index values as the sample submission/test set. No thresholding is done because the competition asks for an illicit probability.

In [ ]:
check = pd.read_csv("submission.csv")
print(check.shape)
print(check.dtypes)
print("Index matches test:", check["index"].equals(test["index"]))
print("Target summary:")
print(check["target"].describe())
